## Baseline scores on semart using CLIP

In [4]:
%load_ext autoreload
%autoreload 2

import os
import json
import torch
import torch.nn.functional as F

import open_clip 
import numpy as np
import pandas as pd


from typing import List
from tqdm import tqdm 
from src.utils import seed_everything, GraphEdgeDataset, process_batch
from src.model import SheafMultimodalGNN
from src.data import get_clip_embedder, build_graph_from_json, load_json_data
from torch_geometric.data import DataLoader
from src.metrics import compute_bidirectional_metrics, get_top_k_recommendations

triplets = '../artistic_sheaf/data/triplets_semart_test.json'
loaded_data = load_json_data(triplets)
print(f"Loaded {len(loaded_data)} triplets from {triplets}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loaded 4569 triplets from ../artistic_sheaf/data/triplets_semart_test.json


In [2]:
## functions

def get_sim_matrix(image_names, text_names, image_embeddings, text_embeddings, img_to_idx, txt_to_idx):
    """
    Compute similarity matrix between image and text embeddings.
    """
    img_to_emb = {name: emb for name, emb in zip(image_names, image_embeddings)}
    txt_to_emb = {name: emb for name, emb in zip(text_names, text_embeddings)} #.squeeze(0)
    
    idx_to_img = {v: k for k, v in img_to_idx.items()}
    idx_to_txt = {v: k for k, v in txt_to_idx.items()}
    
    # 4. Reorder embeddings according to adjacency order
    reordered_img_emb = np.array([img_to_emb[idx_to_img[i]] for i in range(len(img_to_idx))])
    reordered_txt_emb = np.array([txt_to_emb[idx_to_txt[j]] for j in range(len(txt_to_idx))])
    
    # 5. Normalize and compute cosine similarity
    reordered_img_emb /= np.linalg.norm(reordered_img_emb, axis=1, keepdims=True)
    print(f"Reordered image embeddings shape: {reordered_img_emb.shape}")
    reordered_txt_emb /= np.linalg.norm(reordered_txt_emb, axis=1, keepdims=True)
    print(f"Reordered text embeddings shape: {reordered_txt_emb.shape}")
    sim_matrix = reordered_img_emb @ reordered_txt_emb.T
    return sim_matrix


def make_adj_matrix(triplets, field='item2'):
    """
    """
    # Separate unique images (item1) and texts (item2)
    images = sorted({t["item1"] for t in triplets})
    texts = sorted({t[field] for t in triplets})

    # Create mapping
    img_to_idx = {img: i for i, img in enumerate(images)}
    txt_to_idx = {txt: j for j, txt in enumerate(texts)}

    # Initialize adjacency matrix
    adj_matrix = np.zeros((len(images), len(texts)), dtype=int)

    # Fill matrix
    for t in triplets:
        i = img_to_idx[t["item1"]]
        j = txt_to_idx[t[field]]
        adj_matrix[i, j] = 1

    return adj_matrix, img_to_idx, txt_to_idx

def compute_image_to_text_accuracy(sim_matrix: np.ndarray, adj_matrix: np.ndarray) -> float:
    """
    Computes the accuracy of retrieving the correct text for each image
    based on the similarity matrix.

    Args:
        sim_matrix (np.ndarray): [num_images, num_texts] similarity matrix.
        adj_matrix (np.ndarray): [num_images, num_texts] adjacency matrix 
                                 where 1 indicates the correct match.

    Returns:
        float: Accuracy (in range [0, 1])
    """
    # Get the text index with the highest similarity for each image
    top_text_indices = np.argmax(sim_matrix, axis=1)  # [num_images]

    # Check if the top prediction is correct
    correct = adj_matrix[np.arange(sim_matrix.shape[0]), top_text_indices]

    # Compute accuracy
    accuracy = correct.mean()
    return float(accuracy)


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'mps'
print(f"Using device: {device}")
seed_everything(seed=42)

# Load tokenizer and preprocessing
tokenizer = open_clip.get_tokenizer('ViT-B-32')
_, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
# Initialize the model
model = SheafMultimodalGNN(
    latent_dim=512,
    edge_attr_dim=512,
    num_layers=3,
    step_size=1.0,
    lr=1e-4,
    device='cuda' if torch.cuda.is_available() else 'mps'
)
    
# Load checkpoint
checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=18-val_loss=3.03.ckpt", map_location=device)
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)
model.eval()

Using device: mps


SheafMultimodalGNN(
  (clip_model): CLIP(
    (visual): VisionTransformer(
      (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
      (patch_dropout): Identity()
      (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): ModuleList(
          (0-11): 12 x ResidualAttentionBlock(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
            )
            (ls_1): Identity()
            (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=768, out_features=3072, bias=True)
              (gelu): GELU(approximate='none')
              (c_proj): Linear(in_features=3072, out_features=768, bias=True)
            )
            (ls_2): Identity()
          )


In [3]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data, preprocess, tokenizer, base_folder='../SemArt/')
test_graph_data = test_graph_data.to(device)
print("Loaded test data with {} nodes.".format(len(test_node_to_id.keys())))
print("Creating data loaders...")
test_dataset = GraphEdgeDataset(test_graph_data)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)



100%|██████████| 4569/4569 [00:09<00:00, 457.37it/s]


Loaded test data with 5326 nodes.
Creating data loaders...


/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [4]:
# clip_texts = get_clip_texts(loaded_data, 'item2', get_tokenizer('ViT-B-32'), model)
for batch in test_loader:
    x_img, x_text, edge_index, edge_attr = process_batch(batch, 'test')
    embeddings = model(x_img, x_text, edge_index, edge_attr)
    clip_images = F.normalize(embeddings[edge_index[0, :]], dim=1)
    clip_texts = F.normalize(embeddings[edge_index[1, :]], dim=1)

    print(f"Extracted {len(clip_texts)} text embeddings, each of shape {clip_texts[0].shape}")

tensor(0, device='mps:0') tensor(5325, device='mps:0')
Extracted 4569 text embeddings, each of shape torch.Size([512])


In [5]:
clip_images = clip_images.cpu().detach().numpy()
clip_texts = clip_texts.cpu().detach().numpy()

### Image-to-text retrieval	
### Text-to-image retrieval		
r@1	r@5	r@10	

In [7]:
adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data)
print(f"Adjacency matrix shape: {adj_matrix.shape}")

Adjacency matrix shape: (963, 4363)


In [9]:
sim_matrix = get_sim_matrix([t["item1"] for t in loaded_data], 
                            [t["item2"] for t in loaded_data], 
                            clip_images, clip_texts,
                            img_to_idx, txt_to_idx)
print(f"Similarity matrix shape: {sim_matrix.shape}")

Reordered image embeddings shape: (963, 512)
Reordered text embeddings shape: (4363, 512)
Similarity matrix shape: (963, 4363)


In [11]:
compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10])

{'t2i_precision@1': tensor(0.1038),
 't2i_recall@1': tensor(0.0261),
 't2i_ndcg@1': tensor(0.1038),
 't2i_precision@5': tensor(0.0704),
 't2i_recall@5': tensor(0.0865),
 't2i_ndcg@5': tensor(0.0946),
 't2i_precision@10': tensor(0.0504),
 't2i_recall@10': tensor(0.1216),
 't2i_ndcg@10': tensor(0.1046),
 'i2t_precision@1': tensor(0.0621),
 'i2t_recall@1': tensor(0.0620),
 'i2t_ndcg@1': tensor(0.0621),
 'i2t_precision@5': tensor(0.0292),
 'i2t_recall@5': tensor(0.1453),
 'i2t_ndcg@5': tensor(0.1048),
 'i2t_precision@10': tensor(0.0212),
 'i2t_recall@10': tensor(0.2112),
 'i2t_ndcg@10': tensor(0.1262),
 'mean_precision@1': tensor(0.0830),
 'mean_recall@1': tensor(0.0441),
 'mean_ndcg@1': tensor(0.0830),
 'mean_precision@5': tensor(0.0498),
 'mean_recall@5': tensor(0.1159),
 'mean_ndcg@5': tensor(0.0997),
 'mean_precision@10': tensor(0.0358),
 'mean_recall@10': tensor(0.1664),
 'mean_ndcg@10': tensor(0.1154)}

### Retrieval per type of relationship

In [12]:
for typ in list(set([l['link'] for l in loaded_data])):
    print(f"Processing type: {typ}")
    loaded_data_new = [l for l in loaded_data if l['link'] == typ]
    print(f"Loaded {len(loaded_data_new)} triplets for type {typ}")
    adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new)
    print(f"Adjacency matrix shape: {adj_matrix.shape}")
    sim_matrix = get_sim_matrix([t["item1"] for t in loaded_data_new], 
                            [t["item2"] for t in loaded_data_new], 
                            clip_images, clip_texts,
                            img_to_idx, txt_to_idx)
    print(f"Similarity matrix shape: {sim_matrix.shape}")
    print(compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10]))
    

Processing type: description
Loaded 963 triplets for type description
Adjacency matrix shape: (963, 960)
Reordered image embeddings shape: (963, 512)
Reordered text embeddings shape: (960, 512)
Similarity matrix shape: (963, 960)
{'t2i_precision@1': tensor(0.0519), 't2i_recall@1': tensor(0.0519), 't2i_ndcg@1': tensor(0.0519), 't2i_precision@5': tensor(0.0336), 't2i_recall@5': tensor(0.1682), 't2i_ndcg@5': tensor(0.1116), 't2i_precision@10': tensor(0.0248), 't2i_recall@10': tensor(0.2482), 't2i_ndcg@10': tensor(0.1377), 'i2t_precision@1': tensor(0.0333), 'i2t_recall@1': tensor(0.0333), 'i2t_ndcg@1': tensor(0.0333), 'i2t_precision@5': tensor(0.0277), 'i2t_recall@5': tensor(0.1380), 'i2t_ndcg@5': tensor(0.0857), 'i2t_precision@10': tensor(0.0224), 'i2t_recall@10': tensor(0.2234), 'i2t_ndcg@10': tensor(0.1165), 'mean_precision@1': tensor(0.0426), 'mean_recall@1': tensor(0.0426), 'mean_ndcg@1': tensor(0.0426), 'mean_precision@5': tensor(0.0307), 'mean_recall@5': tensor(0.1531), 'mean_ndcg@5

## Zero shot classification

In [5]:
annotations = '../SemArt/semart_test.csv'
df = pd.read_csv(annotations, sep='\t', encoding='latin1')
df.head(), df.shape

(           IMAGE_FILE                                        DESCRIPTION  \
 0  41294-10ladisl.jpg  Of the Hungarian kings St Ladislas is perhaps ...   
 1   42791-1sacris.jpg  This ceiling painting in the sacristy of San S...   
 2   14376-worship.jpg  In the same period when the most talented arti...   
 3  24776-annuncia.jpg  Based on its style the Annunciation is attribu...   
 4  23845-3manet04.jpg  The 1870s were rich in female models for Manet...   
 
                       AUTHOR                             TITLE  \
 0  UNKNOWN MASTER, Hungarian  Saint Ladislaus, King of Hungary   
 1            VERONESE, Paolo          Coronation of the Virgin   
 2         FRANCKEN, Frans II        Worship of the Golden Calf   
 3         MASTER of Flémalle                      Annunciation   
 4             MANET, Edouard        Brunette with Bare Breasts   
 
                       TECHNIQUE     DATE       TYPE     SCHOOL  TIMEFRAME  
 0   Oil on wood, 103 x 101,3 cm  c. 1600  religious  H

In [6]:
loaded_data_new = []
for itm in df['IMAGE_FILE']:
    loaded_data_new.append({})
    loaded_data_new[-1]['item1'] = 'Images/' + itm
    author = df[df['IMAGE_FILE'] == itm]['AUTHOR'].values[0]
    loaded_data_new[-1]['author'] = f"Artwork by {author}"
    timeframe = df[df['IMAGE_FILE'] == itm]['TIMEFRAME'].values[0]
    loaded_data_new[-1]['timeframe'] = f"Artwork painted in {timeframe}"
    school = df[df['IMAGE_FILE'] == itm]['SCHOOL'].values[0]
    loaded_data_new[-1]['school'] = f"Artwork from the {school} school"
    material = df[df['IMAGE_FILE'] == itm]['TECHNIQUE'].values[0].split(',')[0]
    loaded_data_new[-1]['material'] = f"Artwork made with {material}"
    genre = df[df['IMAGE_FILE'] == itm]['TYPE'].values[0]
    loaded_data_new[-1]['genre'] = f"Artwork of the {genre} genre"
    loaded_data_new[-1]['link'] = 'metadata'

print(f"Updated loaded_data with authors, total items: {len(loaded_data_new)}")

Updated loaded_data with authors, total items: 1069


In [7]:
loaded_data_new[0].keys()

dict_keys(['item1', 'author', 'timeframe', 'school', 'material', 'genre', 'link'])

In [8]:
item2 = 'school' #author, timeframe, school, material, genre

In [9]:
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data_new, preprocess, tokenizer, 
                                                                           base_folder='../SemArt/', item2=item2)
test_graph_data = test_graph_data.to(device)
print("Loaded test data with {} nodes.".format(len(test_node_to_id.keys())))
print("Creating data loaders...")
test_dataset = GraphEdgeDataset(test_graph_data)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

# clip_texts = get_clip_texts(loaded_data, 'item2', get_tokenizer('ViT-B-32'), model)
for batch in test_loader:
    x_img, x_text, edge_index, edge_attr = process_batch(batch, 'test')
    embeddings = model(x_img, x_text, edge_index, edge_attr)
    clip_images_cls = F.normalize(embeddings[edge_index[0, :]], dim=1).cpu().detach().numpy()
    clip_texts_author = F.normalize(embeddings[edge_index[1, :]], dim=1).cpu().detach().numpy()

    print(f"Extracted {len(clip_texts_author)} text embeddings, each of shape {clip_texts_author[0].shape}")


100%|██████████| 1069/1069 [00:10<00:00, 104.70it/s]


Loaded test data with 1092 nodes.
Creating data loaders...


/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


tensor(0, device='mps:0') tensor(1091, device='mps:0')
Extracted 1069 text embeddings, each of shape (512,)


In [10]:
adj_matrix_author, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new, field=item2)
sim_matrix_author = get_sim_matrix([t["item1"] for t in loaded_data_new], 
                            [t[item2] for t in loaded_data_new], 
                            clip_images_cls, clip_texts_author,
                            img_to_idx, txt_to_idx)

print(f"Adjacency matrix shape: {adj_matrix_author.shape}", 
      f"Similarity matrix shape: {sim_matrix_author.shape}")

Reordered image embeddings shape: (1069, 512)
Reordered text embeddings shape: (23, 512)
Adjacency matrix shape: (1069, 23) Similarity matrix shape: (1069, 23)


In [23]:
adj_matrix_author[10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0])

In [24]:
sim_matrix_author[10].argmax()

np.int64(13)

In [11]:
compute_image_to_text_accuracy(sim_matrix_author, adj_matrix_author)

1.0

In [29]:
txt_to_idx

{'Artwork from the American school': 0,
 'Artwork from the Austrian school': 1,
 'Artwork from the Belgian school': 2,
 'Artwork from the Bohemian school': 3,
 'Artwork from the Danish school': 4,
 'Artwork from the Dutch school': 5,
 'Artwork from the English school': 6,
 'Artwork from the Flemish school': 7,
 'Artwork from the French school': 8,
 'Artwork from the German school': 9,
 'Artwork from the Greek school': 10,
 'Artwork from the Hungarian school': 11,
 'Artwork from the Irish school': 12,
 'Artwork from the Italian school': 13,
 'Artwork from the Netherlandish school': 14,
 'Artwork from the Other school': 15,
 'Artwork from the Polish school': 16,
 'Artwork from the Portuguese school': 17,
 'Artwork from the Russian school': 18,
 'Artwork from the Scottish school': 19,
 'Artwork from the Spanish school': 20,
 'Artwork from the Swedish school': 21,
 'Artwork from the Swiss school': 22}

In [12]:
recs = get_top_k_recommendations(torch.Tensor(sim_matrix_author), k=5)

In [30]:
# Print which query gives which recommendation (text or image path)
# For zero-shot classification, queries are image paths (from loaded_data_new), recommendations are text (e.g., timeframe, author, etc.)

query_field = 'item1'  # image path
rec_field = item2      # e.g., 'timeframe', 'author', etc.
idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}

for i, rec_indices in enumerate(recs):
    query = loaded_data_new[i][query_field]
    recommendations = [idx_to_txt[j] for j in rec_indices]
    print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
    print(f"Ground Truth: {loaded_data_new[i][rec_field]}")
    print("Recommendations:")
    for rec in recommendations:
        print(f"  - {rec}")
    print("-" * 40)

Query: /Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/Images/41294-10ladisl.jpg
Ground Truth: Artwork from the Hungarian school
Recommendations:
  - Artwork from the Dutch school
  - Artwork from the Belgian school
  - Artwork from the German school
  - Artwork from the Danish school
  - Artwork from the Austrian school
----------------------------------------
Query: /Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/Images/42791-1sacris.jpg
Ground Truth: Artwork from the Italian school
Recommendations:
  - Artwork from the Dutch school
  - Artwork from the Belgian school
  - Artwork from the German school
  - Artwork from the Danish school
  - Artwork from the Austrian school
----------------------------------------
Query: /Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/Images/14376-worship.jpg
Ground Truth: Artwork from the Flemish school
Recommendations:
  - Artwork from the Netherlandish school
  - Artwork from the Flemish school
  - Artwork from the Dutch school
  - Artwork from the